<!---
Notas de clase Métodos Computacionales
Por 
Óscar Antonio Restrepo Gutiérrez
--->

# Solución de sistemas lineales con numpy y scipy
1) [Matrices reales y complejas](#Matrices_reales_y_complejas).<br>
2) [Comparación de matrices con python](#Comparación_de_matrices_con_python).<br>
3) [Factorización LU y casos especiales](#FactorizacionLU).<br>
4) [Diagonalización](#Diagonalizacion). <br>
5) [Material complementario](#Material_complementario).




NumPy y SciPy utilizan implementaciones optimizadas de librerías de álgebra lineal como ATLAS, MKL, OpenBlas y LAPACK bajo la interfaz de BLAS para realizar operaciones maticiales, estas librerías proporcionan rutinas altamente optimizadas para operaciones comunes de álgebra lineal, así que no implementamos las nuestras y mejor aprendemos a usar estas. ATLAS, OpenBlas y MKL son implementaciones de BLAS, que hacen las operaciones basicas de sumas, restas, multiplicación mientras que LAPACK se basa en BLAS y proporciona funciones más avanzadas, como la descomposición de matrices LU, QR, Cholesky; resolución de sistemas de ecuaciones lineales; diagonalización; etc.

Basado en estas liberias NumPy proporciona estructuras de datos eficientes y operaciones de álgebra lineal fundamentales, SciPy extiende esta capacidades con un conjunto más amplio de funciones, estas librerías se importan como,
```python
import scipy.linalg 
import numpy.linalg
```

Aquí usaremos scipy:

In [ ]:
import numpy as np
import scipy.linalg as LA

<a id='Matrices_reales_y_complejas'></a> 
## 1) Matrices reales y complejas
Una matriz compleja es aquella que tiene números complejos en sus entradas, ejemplo, 

$$
\begin{equation}
A1 =  
\begin{pmatrix}
1   & 1-2j & j \\
1+2j&  1   & 0 \\
-j  &  0   & 2
\end{pmatrix}
\end{equation},
$$

La matriz transpuesta $A^t$ se obtiene al cambiar las filas por las columnas o $a_{ij}\Rightarrow a_{ji}$, en matrices complejas después de transponer hay que conjugar las entradas, ósea, $a_{ij}\Rightarrow  a_{ji}^*$ (donde $a^*_{ji}=x-yj$ es el conjugado de $a_{ji}=x+yj$); la transpuesta conjugada se define como $A^\dagger$. En python la transpuesta (real o compleja) se calcula usando la rutina, 
```python 
numpy.transpose()
```    
Si $A$ es compleja hay que transponer y conjugar usando,
```python
numpy.transpose().conj()
```    
Pero esto es engorroso, más fácil aún es usar, 
```python
A.T (transpuesta, usar np.array() o np.matrix())
A.H (transpuesta conjugada, solo funciona con np.matrix() )
```

Ejemplo:

In [ ]:
# Transpuesta de una matriz real
#A = np.arange(1,10).reshape(3,3)
A = np.array([[1,2,3],[4,5,6],[7,8,9]])
print ('A =\n',A)
print ('\nTranspuesta de A = \n',np.transpose(A)) # o también A.transpose()

In [ ]:
# Transpuesta conjugada de una matriz compleja
A = np.matrix([[1+1j,2+2j,3+3j],[4+4j,5+5j,6+6j],[7+7j,8+8j,9+9j]])
#A = np.arange(1,10) + 1j*np.arange(1,10) # array: real + imag
#A = A.reshape(3,3)

print ('\nA =\n',A)
print ('\nTranspuesta conjugada de A = \n',A.transpose().conj())
# o también
print ('\nTranspuesta A.H de A = \n',A.H)

### El producto punto de vectores complejos 
Se define por,
$$
\begin{align}
\mathbf{u}\cdot\mathbf{v}&=\mathbf{u}^\dagger\mathbf{v}\quad\longrightarrow\quad\text{da complejo},\\
\mathbf{u}\cdot\mathbf{u}&=\mathbf{u}^\dagger\mathbf{u}\quad\longrightarrow\quad\text{da real},
\end{align}
$$

donde $\mathbf{u}^\dagger$ es fila y $\mathbf{v}$ es columna, además note que $\mathbf{v}^\dagger\mathbf{u}=(\mathbf{u}^\dagger\mathbf{v})^*$, da el conjugado osea que no commuta como el producto punto en los reales. El producto punto de un vector consigo mismo define la norma al cuadrado $\mathbf{u}^\dagger\mathbf{u}=\|\mathbf{u}\|^2$ y siempre da un número real, en python se calcula con,
```python
linalg.norm(u) # da raíz del producto punto.
```
Para el producto punto $\mathbf{u}\cdot\mathbf{v}$ en los métodos de numpy, `matmul()`, `dot()` y `@` siempre hay que tomar el conjugado de $\mathbf{u}$:

In [ ]:
u = np.array([1,1+1j])
v = np.array([2+3j,1+1j])

u.conj().dot(v), np.dot(u.conj(), u), v.dot(v.conj()), # La norma debe ser real
#u.dot(v.conj()) # Cambia por el conjugado.

In [ ]:
LA.norm(u)

### Matrices simétricas y hermíticas
Una matriz real es simétrica si es igual a su transpuesta: 
\begin{equation*}
A=A^t \mbox{  o   } a_{ij}=a_{ji}.
\end{equation*}
En el caso de matrices complejas se dice que una matriz es hermítica si es igual a su transpuesta conjugada,
\begin{equation*}
A= A^\dagger \mbox{  o   } a_{ij}=a^*_{ji},
\end{equation*}
Note que la matriz anterior $A1$ es hermítica pues su transpuesta conjugada da otra vez $A1$. Para crear una matriz simétrica  hay tres maneras, 
$$A^tA\quad \text{ o }\quad AA^t\quad \text{ o también }\quad \frac{A+A^t}{2},$$ 

esto también funciona con hermíticas, pero se toma la compleja conjugada.

Ejemplo:


In [ ]:
# Crear matriz simétrica
A = np.arange(1,10).reshape(3,3)

A@A.T, A.T@A, (A + A.T)*.5 # hay tres maneras

In [ ]:
# Crear matriz compleja: parte real más parte imaginaria
n = 3  # dimensión
A_re = np.random.randint(-10,10,size=(n,n)) # parte real
A_im = np.random.randint(-10,10,size=(n,n)) # parte imaginaria
A = A_re + 1j*A_im

A # matriz compleja

In [ ]:
# Crear matriz hermítica, dos maneras:
# Note que la diagonal es real
A@A.T.conj(), (A + A.T.conj())*.5

### Matrices ortogonales y unitarias
En los reales, si la transpuesta de una matriz es igual a su inversa, entonces se dice que la matriz es *ortogonal*, 

$$
\begin{equation*}
A^{-1}=A^t, \iff A^tA=I
\end{equation*}
$$

Propiedades de matrices ortogonales
+   su determinate es $\pm 1$,
+   el producto punto de dos vectores se conserva $(A\mathbf{x})\cdot(A\mathbf{y})=\mathbf{x}^tA^tA\mathbf{y}=\mathbf{x}^t\mathbf{y}=\mathbf{x}\cdot \mathbf{y}=$ const,
+   los vectores formados con sus columnas son unitarios y ortogonales entre sí,
+   la matriz cambio de una base ortonormal (es decir, que sus vectores son unitarios y perpendiculares entre sí) $\bf{\hat i, \hat j, \hat k}$  a otra base ortonormal $\bf{\hat i', \hat j', \hat k'}$ es ortogonal.

Por ejemplo la matriz,

$$
\begin{equation}
A=\begin{pmatrix}
0&-0.80&-0.60\\0.80&-0.36&\;\;\,0.48\\0.60&\;\;\,0.48&-0.64
\end{pmatrix}
\end{equation}
$$

es ortogonal. Para solución de ecuaciones note que si $A\mathbf{x}=\mathbf{b}$ entonces $\mathbf{x} = A^{-1}\mathbf{b}=A^t\mathbf{b}$, lo cual simplifica el trabajo. En general, una transformación ortogonal $\mathbf{x}'=A\mathbf{x}$ con determinate 1 es *propia* y se puede interpretar como una rotación del vector $\mathbf{x}$ a $\mathbf{x}'$ (si el determinante es -1 entonces es *impropia* y no es una rotación sino una inversión de los ejes, es decir como mirar en un espejo; una simple permutación de dos columnas o filas transforma de impropia a propia la matriz $A$).


En los complejos, si la transpuesta conjugada de una matriz es igual a su inversa, entonces se dice que la matriz es *unitaria*,

$$
\begin{equation*}
A^{-1}= A^\dagger \iff AA^\dagger=I,
\end{equation*}
$$

Propiedades de matrices unitarias,
+  $|\det A|=1$ pues $\det A=e^{j\phi}$ ($\phi$ un número real cualquiera),
+  en el producto punto de dos vectores se cumple que $(A\mathbf{x})\cdot(A\mathbf{y})=\mathbf{x}^\dagger A^\dagger A\mathbf{y}=\mathbf{x}\cdot \mathbf{y}$,
+  sus columnas (o filas) forman una base ortonormal,
+  $A$ es diagonalizable,
+  sus autovectores son ortonormales,
+  se puede escribir en forma exponencial $A = e^{jH}$, donde $H$ es una matriz hermítica,
+  y ademas $\det e^H = e^{\hbox{traza} (H)}$.

Por ejemplo,

$$
\begin{equation}
A=\begin{pmatrix}
1/\sqrt 2  &1/\sqrt 2 &-0\\
-1/\sqrt 2j&1/\sqrt2 j&-0\\
0          & 0        & j
\end{pmatrix}
\end{equation}
$$

es unitaria.

Es importante identificar cuando se tienen este tipo de matrices, ya que en el caso de sistemas $A\mathbf{x}=\mathbf{b}$, no se necesita usar librerías para hallar $\mathbf{x}$, dado que la solución es encontrada de manera fácil analíticamente, pues $\mathbf{x}=A^\dagger \mathbf{b}$.

**Ejercicio:** use python con las matrices anteriores (y vectores $\mathbf{x}, \mathbf{y}$ y $\mathbf{b}$ aleatorios) para verificar las propiedades de matrices ortogonales y unitarias (excepto las propiedades de diagolalización que se discutirán más adelante en la última sección).


In [ ]:
# Hacer tarea

<a id='Comparación_de_matrices_con_python'></a> 
## Comparación de matrices con python
Cuando comparamos números enteros usamos,`a == b`, pero si son flotantes usamos `|a-b|< eps`, es decir comparamos con cierta tolerancia, por ejemplo `eps = 1e-6`. 
En python se puede comparar matrices del mismo modo pero elemento a elemento, hay varias funciones para esto y dependen de si las matrices tienen valores enteros o flotantes.
Las funciones más comunes para comparar matrices (y arrays) con coeficientes tipo entero son,
```python
(A == B)                 # da matriz con falsos y verdaderos
(A == B).all()           # da falso o verdadero
np.equal(A, B)           # da matriz con falsos y verdaderos
np.array_equal(A, B)     # da falso o verdadero
```
y si los coeficientes son tipo flotante,
```python
(abs(A-B) < 1e-8)        # da matriz con falsos y verdaderos
(abs(A-B) < 1e-8).all()  # da falso o verdadero
np.amax(abs(A-B)) < 1e-8 # da falso o verdadero (amax retorna elem máximo del array)
np.isclose(A, B)         # da matriz con falsos y verdaderos 
np.allclose(A, B)        # da falso o verdadero
```   
Verifiquemos si $A$ es simétrica o es hermítica en los siguientes casos:

In [ ]:
# Verificar si A es simétrica
A = np.array([[1,2,3],
              [4,5,6],
              [7,8,9]])

(A.T == A).all()

In [ ]:
# Verificar si A es es hermítica
A = np.matrix([[1, 1-2j, 1j],
               [1+2j, 1, 0],
               [-1j, 0, 2]])

(A.H == A).all() 
# A.H == A  # Verifique que da quitar .all()

In [ ]:
# Comparación floats
A = np.random.rand(3,3)
B = A + A*1e-5

(abs(A-B) < 1e-4).all(), (abs(A-B) < 1e-6).all()

<a id='FactorizacionLU'></a> 
## Factorización LU y casos especiales
### Factorización LU caso general
Suponga que una matriz se puede expresar como el producto de dos matrices $A = LU$ donde $L$ es una matriz triangular inferior y $U$ es una matriz triangular superior, si este es el caso se puede demostrar que el número de operaciones pasa de $O(n^3/3)$ a $O(2n^2)$. Si $A$ es factorizada como $LU$ entonces,
$$
\begin{align}
  A\mathbf{x} & = \mathbf{b}\\
  L(U\mathbf{x})&= \mathbf{b}
\end{align}
$$
Si se define $\mathbf{y}=U\mathbf{x}$ entonces el problemas se resuelve como,
$$
\begin{align}
L\mathbf{y} &= \mathbf{b}\\
U\mathbf{x} &= \mathbf{y}.
\end{align}
$$
En el caso de que no se conozcan $L$ y $U$ se debe calcular primero estas dos matrices, esta operación tiene el mismo costo que hacer una eliminación gaussiana es decir, $O(n^3/3)$, no obstante una vez conocidas $L$ y $U$, la descomposición $LU$ puede ser aplicada a cualquier valor de $\mathbf{b}$, es en este caso que reside su importancia. Por ejemplo imagine que tiene que calcular $A\mathbf{x}_n=\mathbf{b}_n$ para $n=1,2,..., N$ (otro caso típico es el método iterado $A\mathbf{x}_{n+1}=\mathbf{x}_n$) esta operación tomará un tiempo de $O(N*n^3/3)$, mientras que si se hace $LU\mathbf{x}_n=\mathbf{b}_n$, solo se tendrá que calcular $A=LU$ para $n=1$, (con un tiempo aproximado de $O(n^3/3)$), una vez calculada $LU$ los demás pasos solo tomarán un tiempo de $O(2(N-1)n^2)$. 

No todas las matrices no singulares se pueden descomponer en la forma $LU$, pero si se usan permutaciones se puede demostrar que $PA=LU$, donde $P$ es la matriz de permutación formada a partir de intercambiar las filas de la matriz identidad, por ejemplo,

$$
\begin{equation}
PA=\begin{pmatrix}
 0&1& 0\\
 1&0& 0\\
 0&0& 1
\end{pmatrix}
\begin{pmatrix}
 a&b& c\\
 A&B& C\\
 g&h& i
\end{pmatrix}
=\begin{pmatrix}
 A&B& C\\
 a&b& c\\
 g&h& i
\end{pmatrix}
=LU=\begin{pmatrix}   
   1      & 0      & 0 \\
   L_{21} & 1      & 0 \\
   L_{31} & L_{32} & 1\\
\end{pmatrix}
\begin{pmatrix}   
   U_{11} & U_{21} & U_{31} \\
        0 & U_{22} & U_{32} \\
        0 & 0      & U_{33}
\end{pmatrix} \\
\end{equation}
$$

Note que la diagonal de $L$ son unos. En este caso la factorization es de la forma $A = P^{-1}LU = P^tLU$ ($P$ tiene la propiedad que su inversa es igual a su transpuesta).
En scypy.linalg la rutina que hace la descomposición factorial es:

    P, L, U = scipy.linalg.lu(A)
La salida es una tupla con 3 matrices.

Ejemplo:

In [ ]:
import numpy as np
import scipy.linalg as LA
M = np.array([[7, 3, -1, 2], 
           [3, 8, 1, -4], 
           [-1, 1, 4, -1], 
           [2, -4, -1, 6] ])

P, L, U = LA.lu(M)

print("\n P = \n",P)
print("\n L = \n",L)
print("\n U = \n",U)

El sistema $A\mathbf{x}=\mathbf{b}$ se resuelve en dos pasos, 1) se calcula la descomposición $LU$ con pivote de la matriz, la cual es $A = P^tLU$ y el valor se asigna a una nueva variable,

     lu = scipy.linalg.lu_factor(A)
2) luego resolvemos el sistema para $lu$

     x = scipy.linalg.lu_solve(lu,b)
Ejemplo:


In [ ]:
# lu_factor retorna tupla lu = (LU, piv)
# LU:  matriz que almacena U en triángulo superior y L en triángulo inferior 
#      (los unos diagonales de L no se almacenan).
# piv: array, índices de pivote que representan la matriz de permutación P:
#      la fila i de la matriz se intercambió con la fila piv[i].

b  = np.array([1,1,1,1])
lu = LA.lu_factor(M)   
x  = LA.lu_solve(lu,b) # resuelva M x = b a partir de lu_factor: lu_solve((LU,piv), b).
print ("Valor usando LU: x =",x)
print ("Usando solve   : x =", LA.solve(M,b))

Note que como $L$ y $U$ son triangulares y todos los elementos de $L$ son uno, entonces el determinante es simplemente el producto de los elementos de la diagonal de $U$, ósea, 

$$\det A = \det P^t\det L\det U = (-1)^k\det U = (-1)^k\prod_{i=1}^n U_{ii}.$$

donde el determinate de $P$ es $(-1)^k$ y $k$ es el número de permutaciones.


**Definición**: una matriz es estrictamente diagonal dominate cuando se cumple que
$$
\begin{equation*}
|a_{i,i}| > \sum^n_{\substack{j=1\\ j\neq i}} |a_{i,j}|,\quad \forall i=\{1,...,n\}.
\end{equation*}
$$

Entonces en este caso se puede hacer eliminación gaussiana y el sistema $A\mathbf{x}=\mathbf{b}$ tiene solución estable y no se requiere intercambio de filas o columnas.

### Matriz definida Positiva
Una matriz se define como positiva si para cualquier valor de $\mathbf{x}$ se cumple que $\mathbf{x}^tA\mathbf{x}>0$.
Propiedades:
+  $A$ tiene inversa.
+  $a_{ii}>0$.
+  $(a_{i j})^2 < a_{ii}a_{jj}$ , para cada  $i \neq j$.
+  El determinante de cada uno de los menores principales es mayor que cero.
+  Se puede hacer eliminación gaussiana sin intercambio de filas o columnas.
+  Si $A$ es simétrica, una condición suficiente para que sea definida positiva es que tenga elementos diagonales positivos y sea diagonalmente dominante.
+  Si $A$ es simétrica, existe una matriz invertible $B$ tal que $A=B^tB$ (o también $A=BB^t$), pues  $x^t\!Ax = (Bx)^t Bx > 0$ para $x\ne 0$. 

Esta última propiedad nos sirve para construir matrices definidas positivas a partir de matrices cualquiera $B$, por ejemplo $B$ puede una matriz con entradas aleatorias: `B=np.random.random(n,n)`. Note que no todas las matrices definidas positivas deben ser simétricas, solo se requiere que cumpla $\mathbf{x}^tA\mathbf{x}>0$, por ejemplo

$$
(x\ y)\left(\matrix{1&1\cr -1&1}\right)\left(\matrix{x \cr y}\right)=(x\ y)\left(\matrix {x+y\cr-x+y}\right)=(x^2+xy)+(-xy+y^2)=x^2+y^2 > 0.
$$
También note si se usa $A=(B+B^t)/2$, $A$ no es necesariamente definida positiva aunque si simétrica, por ejemplo la matriz $B=\left(\matrix{1&2\cr 3&4}\right)$ da, $(x\ y)\left(\matrix{1&5/2\cr 5/2&4}\right)\left(\matrix{x \cr y}\right)=x^2 + 4y^2 + 5xy$, que puede ser menor que cero.

**Tarea**: a) cree matrices $A$ a partir de matrices $B$ aleatorias de diversos tamaños $n$ y verifique las propiedades anteriores. b) ¿Si ` B=np.arange(n*n).reshape(n,n)` qué concluye?

In [ ]:
# Hacer tarea:

### Descomposición de Cholesky $A=LL^t$
En general, $A$ es una matriz simética y definida positiva si y solo si se puede descomponer en la forma $LL^t$, es decir

\begin{align}
A=LL^t & =
\begin{pmatrix}   
   L_{11} & 0      & 0 \\
   L_{21} & L_{22} & 0 \\
   L_{31} & L_{32} & L_{33}\\
\end{pmatrix}
\begin{pmatrix}   
   L_{11} & L_{21} & L_{31} \\
        0 & L_{22} & L_{32} \\
        0 & 0      & L_{33}
\end{pmatrix} \\
& =
\begin{pmatrix}   
   L_{11}^2     & L_{21}L_{11}              &  L_{31}L_{11}   \\
   L_{21}L_{11} & L_{21}^2 + L_{22}^2       & L_{31}L_{21}+L_{32}L_{22}\\
   L_{31}L_{11} & L_{31}L_{21}+L_{32}L_{22} & L_{31}^2 + L_{32}^2+L_{33}^2
\end{pmatrix}
\end{align}

Esto se conoce como descomposición o factorización de Cholesky. Si $A$ no es definida positiva la descomposición no es única. El algoritmo de Cholesky que es una versión modificada de la eliminación gaussiana, se puede usar para calcular la descomposición matricial $LL^t$, la descomposición matricial de Choleski se hace en scipy con la función,
```python   
scipy.linalg.cholesky()
```
y también se puede aplicar a matrices con complejos, osea, si $A$ es hermítica definida positiva: $A=LL^\dagger$.

Ejemplo

In [ ]:
# en reales
M = np.array([[7, 3, -1, 2], 
              [3, 8, 1, -4], 
              [-1, 1, 4, -1], 
              [2, -4, -1, 6] ])
L = LA.cholesky(M, lower=True)
print ("Cholesky (real) L =\n",L)


# en complejos
A = np.array([[1,-2j],[2j,5]])
L = LA.cholesky(A, lower=True)
print ("\nCholesky (compleja) L =\n",L)

**Ejercicio**: usando python verifique que $M$ cumple las propiedades de una matriz definida positiva.  

### Matriz simétrica $A = LDL^t$;
Si $A$ es simétrica y definida positiva entonces también se puede descomponer en la forma,

\begin{align}
A=LDL^t & =
\begin{pmatrix}   
   1      & 0 & 0 \\
   L_{21} & 1 & 0 \\
   L_{31} & L_{32} & 1\\
\end{pmatrix}
\begin{pmatrix}   
 D_1 & 0 & 0 \\
   0 & D_2 & 0 \\
   0 & 0 & D_3\\
\end{pmatrix}
\begin{pmatrix}
   1 & L_{21} & L_{31} \\
   0 & 1 & L_{32} \\
   0 & 0 & 1\\
\end{pmatrix} \\
& = \begin{pmatrix}   
   D_1       & L_{21}D_1                   & L_{31}D_1   \\
   L_{21}D_1 & L_{21}^2D_1 + D_2           & L_{31}L_{21}D_{1}+L_{32}D_2 \\
   L_{31}D_1 & L_{31}L_{21}D_{1}+L_{32}D_2 & L_{31}^2D_1 + L_{32}^2D_2+D_3
\end{pmatrix}.
\end{align}

Como se puede ver $A$ es simétrica, $L$ es triangular inferior con la diagonal formada por unos y $D$ es una matriz diagonal. El cálculo de Choleski (descomposiciones $LL^t$ o $LDL^t$) toma un tiempo del orden de $t \approx O(n^3/6)$, ósea, aproximadamente la mitad del tiempo de una eliminación gaussiana normal.
Importante, note que la matriz $L$ de la descomposición $A = LDL^t$ no es la misma de $A = LL^t$, pero se cumple que

$$A = {L D L}^t = L D^{1/2} (D^{1/2})^t L^t = L  D^{1/2} (LD^{1/2})^t.$$

La descomposición $LDL^t$ aún no está implementada en *scipy* pero se puede hacer con la rutina de `cholesky()`, si se define $S$ como la matriz diagonal que  contiene los elementos de la diagonal principal de $L^{ch}$ en la descomposición de Cholesky, entonces se puede demostrar que,

$$D = S^2\\ L=L^{ch}S^{-1}.$$

La implementación es:

In [ ]:
def ldl(A):
    A = np.asmatrix(A) # si es array, ver como matriz 
    # verificar que A es simétrica o hermítica
    if np.allclose(A.H, A): # (A.H == A).all()
        Lch  = np.linalg.cholesky(A)
        S    = np.diag(np.diag(Lch))
        Sinv = np.diag(1/np.diag(S))
        
        D = np.matrix(S.dot(S))
        L = np.matrix(Lch.dot(Sinv))      
            
        return L, D
    else:
        print("A debe ser Simétrica or Hermítica.")
        return None, None

    
M = np.array([[7, 3, -1, 2], 
              [3, 8, 1, -4], 
              [-1, 1, 4, -1], 
              [2, -4, -1, 6] ])
 
L, D = ldl(M)
print("L =\n",L)
print("\nD = \n",D)

# recuperar matriz original:
print ("\nLDL^t =\n", L*D*L.H)

### Matriz tridiagonal  y matrices de bandas

Cuando $A$ es una matriz tridiagonal el sistema $A\mathbf{x}=\mathbf{d}$ tiene la forma,

$$
\begin{equation*}
a_i x_{i - 1}  + b_i x_i  + c_i x_{i + 1}  = d_i
\end{equation*}
$$

o en representación matricial,

$$
\begin{equation*}
\begin{pmatrix}
   {b_1} & {c_1} & {   }  & {   }  & { 0 } \\
   {a_2} & {b_2} & {c_2}  & {   }  & {   } \\
   {   } & {a_3} & {b_3}  & \ddots & {   } \\
   {   } & {   } & \ddots & \ddots & {c_{n-1}}\\
   { 0 } & {   } & {   }  & {a_n}  & {b_n} \\
\end{pmatrix}
\begin{pmatrix}
   {x_1 }  \\
   {x_2 }  \\
   {x_3 }  \\
   \vdots  \\
   {x_n }  \\
\end{pmatrix} 
=
\begin{pmatrix}
   {d_1 }  \\
   {d_2 }  \\
   {d_3 }  \\
   \vdots  \\
   {d_n }  \\
\end{pmatrix}
\end{equation*}
$$

Si $A$ es tridiagonal la factorización $LU$ toma una forma bastante simple:
$L$ tiene unos en su diagonal principal y ceros en las otras partes excepto
en la diagonal inmediatamente arriba de la diagonal principal. $U$ tiene sus únicas
entradas distintas de cero en la diagonal abajo de la diagonal principal ([ver problema](#problema10)). Por lo tanto este algoritmo 
solo requiere $(5n − 4)$ multiplicaciones/divisiones and $(3n − 3)$ sumas o restas, lo cual indica que es muchísimo más rápido que usar métodos estándar que son del orden de $\frac{n^3}{3}$ (si hay más diagonales distintas de cero, la matriz se denomina una *matriz de bandas*).

En física estas matrices son comunes en casos como, esplines cúbicos en interpolación, el método de aproximación del enlace fuerte ("tight binding" en inglés) en mecánica cuántica y el método de Crank-Nicolson para solución de ecuaciones diferenciales parciales etc.

Para resolver el sistema de ecaciones $A\mathbf{x} = \mathbf{d}$, python da el método,

```python
scipy.linalg.solve_banded, 
```
veamos:

In [ ]:
import numpy as np
import scipy.linalg as LA
# Diagonales de la matriz triangular:
c = [0, 3, 3, 3, 5]    # Diagonal superior, primer elem debe ser cero
b = [8, 2, 8, 3, 9]    # Diagonal central 
a = [1, 1, 3, 3, 0]    # Diagonal inferior, último elem debe ser cero

d = [1, 1, 1, 1, 1]
# crear matriz tridiagonal nxn 
A = np.diag(c[1:], 1) + np.diag(b, 0) + np.diag(a[:-1], -1)

print ("Matriz tridiagonal A =\n", A )

# Resolver Ax = d, usando scipy.linalg.solve_banded. 
# ab[u + i - j, j] == a[i,j], donde (u,l) son el numero diagonales 
# por encima y debajo de la principal, para tridiagonal es (1,1).
ab = np.array([c,b,a]) # matriz 3x5
x = LA.solve_banded ((1,1),ab,d) # Ax = d <==> LUx = d
print ("\nSolución Ax = d con A tridiagonal:\n x =",x)
print ("\nSolución Ax = d usando solve:\n x =",LA.solve(A,d))

In [ ]:
# Comparar tiempos de computo para matriz tridiagonal aleatoria
n = 100
D = np.random.rand(3,n)
d = np.ones(n)
A = np.diag(D[0,1:], 1) + np.diag(D[1], 0) + np.diag(D[2,:-1], -1)

%timeit LA.solve_banded ((1,1),D,d) # Ax = d <==> LUx = d
%timeit LA.solve(A,d)

<a id='Diagonalizacion'></a> 
## Diagonalización
El problema de diagonalización está relacionado al problema de encontrar la solución al sistema de ecuaciones de la forma, $$A\mathbf{x}=\lambda \mathbf{x},$$ donde $\lambda$ es un escalar, en este caso no podemos utilizar los métodos anteriores, pues $\mathbf{b}=\lambda \mathbf{x}$ también es desconocido.
La diagonalización matricial consiste en transformar una matriz $A$ en otra matriz $D$ diagonal que tiene las mismas propiedades que la matriz original, es decir la diagonalización es equivalente a transformar un sistema de ecuaciones en otro conjunto en el cual la matriz toma una forma canónica, ósea $$A=UDU^{-1}.$$
En otras palabras, se dice que una matriz es diagonalizable si existe una matriz $U$ tal que $$D=U^{-1}AU,$$ donde $D$ es diagonal, si esto se cumple se dice que $A$ es similar a $D$. 
Si la matriz $U$ es ortogonal caso que se da cuando $A$ es simétrica, físicamente la diagonalización de la matriz se interpreta como una rotación de sus ejes de tal manera que estos queden alineados con sus autovectores. 

### Aplicaciones de diagonalización
+ **Matemáticas**: cálculo de potencias de matrices $A^k$ y en general de funciones de matrices $f(A)$, solución de sistemas de ecuaciones diferenciales etc, pues $f(A) =U f(D) U^{-1}.$ 
+ **Física**: cálculo del tensor de inercia, osciladores acoplados, circuitos de elementos pasivos simples, mecánica cuántica (cualquier cantidad que pueda medirse en un experimento físico se asocia con un operador hermítico, por ejemplo, el operador de energía se llama hamiltoniano y es representado por una matriz hermítica. Cuando diagonalizas el hamiltoniano, en la diagonal principal obtendrás las energías del sistema).
+ **Astronomía**: rotación de objetos astronomicos tales como asteroides, achatamiento de planetas, [velocidad de dispersión](https://en.wikipedia.org/wiki/Velocity_dispersion) de grupos de objetos tales como cúmulos abiertos, cúmulos globulares, galaxias o cúmulos de galaxias (la velocidad de dispersión se mide respecto a la velocidad media de un cúmulo, es decir se miden las velocidades radiales de los miembros del grupo mediante técnicas de espectroscopía, una vez obtenida la velocidad de dispersión de ese grupo se puede usar para derivar la masa del grupo), etc.
+ **Química**: la velocidad a la que cambia la concentración de un reactivo es proporcional a su concentración y la concentración de otro reactivo.
+ **Biología**: la velocidad a la que cambian las poblaciones de un depredador y sus presas se resuelve por sistema de ecuaciones donde hay que diagonalizar.

Ver más [aquí](https://www.researchgate.net/profile/Tadeusz-Ostrowski/post/What-are-the-applications-of-Diagonalization-of-a-matrix/attachment/59d6262379197b80779846df/AS%3A320696398352387%401453471387821/download/Applications+of+diagonalization.pdf).


<!---
The diagonalization of a matrix can be interpreted as a rotation of the axes to align them with the eigenvectors.
--->


### Procedimiento de diagonalización 
La diagonalización de una matriz $A$ de tamaño $n\times n$ en un campo $\mathbb{K}^n$ real o complejo se hace en dos pasos:

1)   Se define la ecuación característica $$f(\lambda)=\hbox{det}(A - \lambda I)=0,$$ 
esta ecuación da un polinomio de grado $n$, 
$$f(\lambda) = (\lambda - \lambda_1)^{n_1}(\lambda - \lambda_2)^{n_2}\cdots (\lambda - \lambda_k)^{n_k}$$
donde $k\le n$ y las raices $\lambda_i$ son los autovalores y con ellas se forma la matriz $D$, ademas $n=\sum^k_{i=1}n_i$. En las raíces repetidas se dice que hay "*degeneración*" de orden $n_i$, pero si todos los $n_i=1$ entonces no hay degeneración y hay $n$ raíces distintas.

2) Se remplaza cada raiz en la ecuación matricial $(A-\lambda_i I)\mathbf{x}=0$ y se resuelve para $\mathbf{x}$, 
si el autovalor no es degenerado se obtiene que, 
$$\mathbf{x} = \alpha_1 \mathbf{u}_1,$$
donde $\alpha_1$ es un valor cualquiera, pero si hay degeneración de orden $n_i$ para el autovalor $\lambda_i$, se obtiene que el vector $\mathbf{x}$ encontrado se debe dejar descomponer como una combinación lineal de $n_i$ vectores linealmente independientes, 
$$\mathbf{x} = \alpha_1 \mathbf{u}_1 + \alpha_2 \mathbf{u}_2 + \cdots + \alpha_{n_i} \mathbf{u}_{n_i},$$
con $\alpha_1, \alpha_2 \cdots, \alpha_{n_i}$ valores cualquiera.
Si esto no se cumple, quedan faltando columnas, $\mathbf{u}_{i}$, distintas de cero para formar $U$ y el sistema no es diagonalizable. Los valores $\alpha_i$ pueden ser cualquier número y desaparecen al normalizar el vector.
En un sistema diagonalisable se deben encontrar $n$ autovectores $\mathbf{u}_i$, $i=1,2,\cdots ,n$ linealmente independientes y normalizados (${\bf\hat u}={\bf u}/\|{\bf u}\|$), y con ellos se construyen las columnas de la matriz,

$$
U = \begin{pmatrix}
\mid & \mid & & \mid \\
{\bf\hat u}_{1} & {\bf\hat u}_{2} & \cdots & {\bf \hat u}_{n}\\
\mid & \mid & & \mid \\
\end{pmatrix},
$$
donde los autovalores forman la matriz,
$$
 \begin{align} 
 D = \begin{pmatrix}
 \lambda_1 & 0 & \cdots &0 \\ 0 & \lambda_2 & 0 & 0\\ \vdots & 0 & \ddots & \vdots \\ 0 & 0 & \cdots & \lambda_n 
 \end{pmatrix}
\quad\text{ y si }\lambda_i\neq 0\quad\Rightarrow\quad
 D^{-1} = 
 \begin{pmatrix}
 \frac{1}{\lambda_1} & 0 & \cdots &0 \\ 0 & \frac{1}{\lambda_2} & 0 & 0\\ \vdots & 0 & \ddots & \vdots \\ 0 & 0 & \cdots & \frac{1}{\lambda_n}
 \end{pmatrix} 
 \end{align}.
$$


<!---
#### Solución de sistemas $\boldsymbol{Ax=b}$ con diagonalización
Primero, consideremos el caso en que $A=D$ fuera diagonal, entonces resolver un sistema de ecuaciones $x=D^{-1}b$ sería trivial, pues la inversa de $D$ es,

$$
 \begin{align} 
 D = \begin{pmatrix}
 d_1 & 0 & \cdots &0 \\ 0 & d_2 & 0 & 0\\ \vdots & 0 & \ddots & \vdots \\ 0 & 0 & \cdots & d_n 
 \end{pmatrix}
\iff 
 D^{-1} = 
 \begin{pmatrix}
 \frac{1}{d_1} & 0 & \cdots &0 \\ 0 & \frac{1}{d_2} & 0 & 0\\ \vdots & 0 & \ddots & \vdots \\ 0 & 0 & \cdots & \frac{1}{d_n}
 \end{pmatrix} 
 \end{align}
$$ 

así pues la multiplicación por un vector $b$ solo requiere $n$ operaciones.

En general, si $A=UDU^{-1}$ entonces se puede usar diagonalización para resolver el problema matricial, pues se puede escribir $Ax=b$ como,

$$
\begin{align}
(UDU^{-1})x = &\, (UU^{-1})b\\
UD(U^{-1}x) = &\, U(U^{-1}b)\\
\end{align}
$$

si se define $x'=U^{-1}x$ y $b'=U^{-1}b$, se obtiene la forma canónica (estándar),

$$\boxed{Dx'= b'}$$

esta última ecuación proporciona una manera de canonizar (estandarizar) un sistema en la forma más simple al reducir el sistema de $n\times n$ parámetros a solo $n$ y que conserva las propiedades de la matriz inicial. Finalmente note que,

$$x=UD^{-1}U^{-1}b.$$

Es importante notar que si los $\lambda_i$ soy muy pequeños, esta solución de $x$ puede tener errores muy grandes.
--->

Desafortunadamente este procedimiento por cálculo de la ecuación característica es impráctico de implementar, en computación, el cálculo de los vectores y valores propios de una matriz se hace por el [algoritmo QR](https://en.wikipedia.org/wiki/QR_algorithm), este algoritmo está considerado entre los 10 más importantes del siglo veinte, en el complemento se muestra como hacer la  [descomposición QR](#Descomposición_QR) de una matriz, por lo tanto usamos la siguiente rutina de scipy (también está definida en numpy) que retorna una tupla con los autovalores $e$ y la matriz $U$ con los autovectores:

    e,U = scipy.linalg.eig(A)

también está la rutina que calcula solo autovalores
   
    scipy.linalg.eigvals(A)


En general si $A$ es diagonalizable:
+   $\det A = \det D = \prod\limits_{i=1}^k{\lambda_i^{n_i}}$.
+   Las columnas de $U$, son linealmente independientes y forman una base para $\mathbb{K}^n$.
+   Si hay degeneración, los autovectores tienen la libertad adicional de rotación, ósea cualquier otra combinación de vectores rotados que comparten el mismo autovalor también son autovectores de $A$.
+   $A$ es invertible, $A^{-1}= UD^{-1}U^{-1}$ y tiene autovalores $1/\lambda_i$, pues si recordamos que $(AB)^{-1} = B^{-1} A ^{-1}$, entonces,
$$A^{-1} = (UDU^{-1})^{-1} = (U^{-1})^{-1}D^{-1}U^{-1} = UD^{-1}U^{-1}.$$
+   los autovectores de $A^{-1}$ son los mismos autovectores de $A$.
+   Las potencias de $A$ se pueden calcular como: $A^k =U D^k U^{-1}$ donde $D^k = $ diag$(\lambda^k_1,\lambda^k_2 ,...,\lambda^k_n)$. 
+   En general $f(A) =U f(D) U^{-1}$, donde $f(D) = $ diag$(f(\lambda_1),f(\lambda_2) ,...,f(\lambda_n))$.
+   Si $A$ es triangular superior o inferior, sus autovalores son los elementos de la diagonal y su inversa tiene autovalores $1/\lambda_i$.
+   Agregar el mismo valor $\alpha$ a todos los elementos diagonales de la matriz $A$, no cambia sus vectores propios y los valores propios se desplazan por $\alpha$: $A\mathbf{x}=\lambda \mathbf{x}\iff (A + \alpha I)\mathbf{x}=(\lambda + \alpha) \mathbf{x}$.

La diagonalización es especialmente útil en el caso de matrices hermíticas y simétricas, pues estas matrices tienen las siguientes propiedades:

Matrices hermíticas:

+   Los autovalores de una matriz hermítica son reales.
+   La matriz $U$ que la diagonaliza $A$ es unitaria, $D=U^\dagger AU$. 
+   Si $A$ es definida positiva ($\mathbf{x}^\dagger A\mathbf{x}>0$) y hermítica, entonces sus autovalores son positivos.

Igual con matrices simétricas en los reales: 

+   Los autovalores de una matriz simétrica son reales. 
+   La matriz $U$ que la diagonaliza $A$ es ortogonal, $D=U^t AU$. 
+   Si $A$ es definida positiva ($\mathbf{x}^t A\mathbf{x}>0$) y simétrica, entonces sus autovalores son positivos.

Para estos casos es mejor usar
  
    e,U = np.linalg.eigh(A)

**Ejemplo: rotación del cuerpo rígido**

Sabemos que la rotación de un cuerpo rígido se define por el momento de inercia $I$, y que este depende de la dirección de rotación, así que podemos calcular propiedades como la energía de rotación o el momento ángular. Si la rotación no está alineada con los ejes principales de rotación el momento de inercia será una matriz conocida como tensor de inercia, diagonalizar el tensor de inercia significa encontrar los ejes principales de rotación (que serán los autovectores de $I$). En general la energía cinética de rotación se calcula por la operación matricial  

$$E_k=\frac{1}{2}\boldsymbol{\omega} I \boldsymbol{\omega},$$ 

y el momento angular como $$\mathbf{L}=I \boldsymbol{\omega},$$

donde $\boldsymbol{\omega}$ es el vector de velocidad angular.
Si se impone la condición de que el momento angular sea paralelo a la velocidad angular $I\boldsymbol{\omega} = \lambda\boldsymbol{\omega}$, se define el problema diagonalización $I_D=U^{-1}IU$, entonces $U$ nos da la transformación al nuevo sistema de referencia donde el momento de inercia será una matriz diagonal $I_D$, lo cual simplifica los cálculos pero no afecta los resultados, 
por ejemplo, considere el cambio de base al sistema primado mediante la transformación dada por $U$ para la siguiente barra (ver figura)

|<img src="../figures/Rotacion_ejes.png" alt="Drawing" style="width: 500px;"/>|
|:--:| 
| *Figura: Rotación de ejes para que coincidan con con los ejes principales de rotación del momento de inercia de una barra rotada.*|

entonces para el momento de inercia tendremos

$$U^{t}\mathbf{L}=U^{t}IU(U^{t}\boldsymbol{\omega})\quad\Longrightarrow\quad \mathbf{L'}=I_D \boldsymbol{\omega}'.$$

donde $\mathbf{L}'$ y $\boldsymbol{\omega}'$ serán el momento angular y la velocidad angular respecto al sistema primado y $U^{t}=U^{-1}$, pues $I$ es simétrica. Se puede demostrar que la energía no cambia por cambiase al sistema primado, osea $E_k=E'_k$ (demostrarlo, se hace de manera similar que con el momento angular). 

**Ejercicio**: a) Usar Python para calcular el momento de inercia para un conjunto de partículas colocadas de manera aleatoria en una barra de tamaño $(1,3)$ cm y masas iguales a uno, b) calcular el momento angular y la energía cinética para cualquier vector $\boldsymbol{\omega}$, c) diagonalizar el tensor de inercia y calcular $\mathbf{L'}, \boldsymbol{\omega'}$ y verificar que $E_k=E'_k$ y que $\mathbf{L'} = I_D \boldsymbol{\omega}'$.
El tensor de inercia de un conjunto de masas $m_i$ y coordenadas $\mathbf{r}_i=(x_i,y_i,z_i)$ relativas a su centro de masa se representa por la matriz simétrica

$$
\begin{align*}
\mathbf{I} = \left(
\begin{array}{lll}
I_{xx} & I_{xy} & I_{xz}\\
I_{xy} & I_{yy} & I_{yz}\\
I_{xz} & I_{yz} & I_{zz}
\end{array}\right),
\end{align*}
$$

donde

$$
\begin{align*}
I_{xx} &= \sum_i m_i(y_i^2 + z_i^2), & \quad I_{yy} &= \sum_i m_i(x_i^2 + z_i^2), & \quad I_{zz} &= \sum_i m_i(x_i^2 + y_i^2),\\
I_{xy} &= -\sum_i m_ix_iy_i, & \quad I_{yz} &= -\sum_i m_iy_iz_i, & \quad I_{xz} &= -\sum_i m_ix_iz_i.
\end{align*}
$$

In [ ]:
# Hacer tarea momento de inercia



**Ejemplo de diagonalización**

Considere la matriz triangular superior,
$$A = \begin{pmatrix} -1&-1&1\\ 0&-2&1\\ 0&0&-1\\ \end{pmatrix}$$
entonces los autovalores se calculan mediante la ecuación característica,

\begin{align}
f(\lambda) =\,  & 
\begin{vmatrix}
 -1-\lambda&     -1   &    1\\
 0         &-2-\lambda&    1\\
 0&0 & -1 -\lambda
\end{vmatrix}
= (-1 -\lambda)^2(-2 - \lambda).
\end{align}

Para calcular los autovectores, primero remplazamos $\lambda_1=-2$, que da el sistema de 2 ecuaciones,

\begin{align} x-y &=0\\ 
                z &=0\\ 
\end{align}

en el cual $x=y$, lo que significa que podemos elegir a $x$ (o $y$) con un valor cualquiera, entonces se fija $x = y=\alpha$, eso da el autovector,

$$
\mathbf{x} = \alpha \begin{pmatrix} 1\\ 1\\ 0\\ \end{pmatrix}=\alpha\mathbf{u}_1
$$

En el caso de $\lambda_2=-1$ hay degeneración dos así que se debe encontrar un $\mathbf{x}=\beta\mathbf{u}_2 + \gamma\mathbf{u}_3$, que sea combinación lineal de los dos autovectores restantes. Al remplazar en la matriz nos da el sistema,

\begin{align} 0-y+z &=0\\ 
              0-y+z &=0\\
              0+0+0 &=0
\end{align}

lo que significa que solo sabemos que $y=z$, y $x$ puede ser cualquier valor, así que escogemos $x=\beta$ y $y=z=\gamma$, entonces, 

$$
\mathbf{x} = \begin{pmatrix}  x\\ y\\ z\\ \end{pmatrix}
             = \begin{pmatrix}\beta\\ \gamma\\ \gamma \\\end{pmatrix}
             = \beta  \begin{pmatrix}  1\\ 0\\ 0\\ \end{pmatrix}
             + \gamma \begin{pmatrix}  0\\ 1\\ 1\\ \end{pmatrix}
             =\beta\mathbf{u}_2 + \gamma\mathbf{u}_3.
$$

Al normalizar los vectores las constantes arbitrarias $\alpha, \beta$ y $\gamma$ desaparecen, 

$$
\mathbf{\hat u}_1 = \frac{1}{\sqrt2} \begin{pmatrix} 1\\ 1\\ 0\\ \end{pmatrix}, 
\mathbf{\hat u}_2 = \begin{pmatrix} 1\\ 0\\ 0\\ \end{pmatrix}, 
\mathbf{\hat u}_3 = \frac{1}{\sqrt2} \begin{pmatrix} 0\\ 1\\ 1\\ \end{pmatrix}.
$$

Estos tres autovectores forman las columnas de la matriz $U$, que diagonaliza $A$. 
<!---
Es importante enfatizar que en el caso de degeneración, debido a la elección arbitraria de los valores $\alpha_i$, pueden haber otras alternativas para expresar los autovectores para crear la matriz $U$.
--->

**Ejemplos**:

In [ ]:
# 1) Matriz general real diagonalizable del ejemplo anterior
# M = np.matrix([[-1,2,1],[6,-1,0],[-1,-2,-1]])
M = np.matrix([[-1,-1,1],
               [0,-2,1],
               [0,0,-1]])
e,U = LA.eig(M)
print ("Matriz:\n",M)
print ("\nAutovalores: \n",e)
print ("\nAutovectores: \n",U)

D=np.diag(e,0)  # Crear matriz diagonal usando los autovalores, esto da "array", 
                # la multiplicacion matricial se debe hacer como: 
                #         np.dot(np.dot(U,D),LA.inv(U)))
D=np.asmatrix(D)# pero si al menos un elemento es "matriz", se puede hacer: 
print ("\nVerificando que UDU^-1 == A:\n",U*D*LA.inv(U))

In [ ]:
# 2) Diagonalización matriz hermítica 
H = np.array([[1, 1-2j, 1j],
              [1+2j, 1, 0],
              [-1j, 0, 2]])
e,U = LA.eig(H)
print ("\n\nMatriz hermitica:\n",H) 
print ("\nAutovalores reales: \n",e) 
print ("\nAutovectores complejos:\n",U)

Para la matriz real, la solución encontrada por numpy es igual a la analítica, pero en general $U$ puede diferir a la solución analítica ¿por qué? (respuesta: volver a leer propiedades de diagonalización).

Para la matriz hermítica aparentemente los autovalores tienen parte compleja, pero un mejor análisis nos deja ver que la parte compleja de cada autovector es del orden de $10^{-16}$, entonces estos se pueden considerar como ceros.

Reimprimamos el resultado con solo 3 cifras significativas:

In [ ]:
# imprimiendo con solo 3 cifras significativas después del punto:
print ("\n\nMatriz hermítica:\n",   np.round(H,3)) # imprima con 3 cifras
print ("\nAutovalores reales: \n",  np.round(e,3)) # después del punto.
print ("\nAutovectores complejos:\n",np.round(U,3)) #

**Ejercicio**: crear un matriz simétrica $A$ y una matriz hermítica $H$ ambas de dimensiones variables, $n=5,10,15$,
calcular sus autovectores y autovalores, luego calcular sus inversas y sus determinantes por diagonalización, verifique en cada caso que $D$ es similar $A$ y $H$ (es decir tienen mismas propiedades). 

In [ ]:
# hacer tarea 

**Ejemplo**: para la matriz Hermítica $H$ del ejemplo anterior, use python para a) calcular la matriz $F=e^{3jH}$, b) muestre que $F$ es unitaria.

Solución: recuerde que en general $f(A) =U f(D) U^{-1}$ además $U^{-1}=U^\dagger$ entonces:

In [ ]:
# a)
D = np.diag(np.exp(3j*e)) # matriz diagonal con autovalores de H
F = U@D@U.T.conj()        # UDU^t, pues U es unitaria porque H es hermítica
#F = U@D@LA.inv(U)        # UDU^-1 también funciona
np.round(F,3)

In [ ]:
# b) 
I = np.round(F@F.T.conj(),3)
I#.real 

**Ejemplo**: mostremos que $\cos^2(H)+\text{sen}^2(H)=I$, da la matriz identidad para cualquier matriz $H$:

In [ ]:
# Tarea: verificar para otras matrices de dimensión n
I = U@np.diag( np.cos(e)**2+np.sin(e)**2 )@U.T.conj()
np.round(I)

No todas las matrices son diagonalizables, por ejemplo considere la matriz,

\begin{align}
A & =
\begin{pmatrix}   
   2 & 1 \\
   0 & 2 
\end{pmatrix}
\end{align}

que tiene un solo autovalor degenerado igual a 2. Los autovectores son iguales a $\mathbf{u_1}=\mathbf{u_2}=(1,0)$ y la matriz diagonal es,

\begin{align}
D = &
\begin{pmatrix} 
2&0\\
0&2
\end{pmatrix} 
= 2I.
\end{align}

Al calcular el producto $UDU^{-1} = 2I \neq A$, vemos que no se recupera $A$, ósea que $A$ no es similar a $D$.

Cuando se resuelve en numpy da:

In [ ]:
A = np.matrix([[2,1],[0,2]])
e,U = LA.eig(A)
print ("Matriz:\n",A)
print ("\nAutovalores: \n",e)
print ("\nAutovectores: \n",U)

D=np.diag(e)     # crear matriz diagonal usando los autovalores 
D=np.asmatrix(D) # convertir "np.array" a "np.matrix"
print ("\n El producto matricial UDU^-1 no es A: \n",U*D*LA.inv(U))      

Solo hay un autovalor degenerado igual a 2 y los dos autovectores son el mismo (pues 4.44089210e-16 se puede considerar cero), así que este sistema no tiene solución. Vemos pues que si hay al menos una fila o columna igual a cero en $U$, entonces $A$ no es diagonalizable. 


### Resumen

Propiedades de matrices expeciales:
   +   Matriz transpuesta $A^t$ y transpuesta conjugada $A^\dagger$;
   +   Matriz simétrica $A=A^t$ (en los reales); 
   +   Matriz hermítica $A= A^\dagger$ (transpuesta conjugada en complejos); 
   +   Matriz ortogonal $A^{-1}=A^t$ (reales);
   +   Matriz unitaria $A^{-1}= A^\dagger$ (complejos).<br>

Factorización LU y casos especiales:
   +   Caso general: matriz  $PA = LU$;
   +   Matriz definida Positiva $A = LL^t$;
   +   Matriz simétrica $A = LDL^t$;
   +   Matriz tridiagonal $A = LU$ (forma de bandas).<br>

Diagonalización: 
   +   Sistema de equaciones $A\mathbf{x}=\lambda \mathbf{x};$
   +   Diagonalización $A=UDU^{-1} \iff D=U^{-1}AU;$
   +   Funciones de matrices $f(A) =U f(D) U^{-1}.$

# Ejercicios

**Tarea 1)**: Use numpy para mostrar que $A^{-1}$ tiene autovalores $1/\lambda_i$ y los mismos autovectores de $A$ (use las matrices anteriores).

**Tarea 2)**: Defina dos matrices cualquiera, una simétrica y otra hermítica, verifique las propiedades mencionadas anteriormente de estas dos matrices. 

**Tarea 3)**: La matriz 
$$A = \begin{pmatrix} 2&0&0\\ 1&1&2\\ 1&-1&4\\ \end{pmatrix}$$

tiene solución analítica (ver Burden) con autovalores $\lambda_1 =3$ y $\lambda_2 = 2$ de multiplicidad 2, los autovectores son, $(0, 1, 1)$, $(0, 2, 1)$ y $(-2, 0, 1)$, compare a la solución de python, use
```python
np.matrix([[2, 0, 0],[1, 1, 2,],[1, -1, 4]])
```
 ¿Por qué dan diferente los autovectores en python?

**Tarea 4)**: Genere varias matrices $M$ aleatorias de $n\times n$ con $a_{ij}$ enteros y compruebe que en verdad $PA=LU$ y que $A=P^tLU$, use 
```python 
A = np.random.randint(1,10, size=(n,n)) # matriz nxn con números aleatorios entre 1 y 10
```       
**Tarea 5)** Repita el ejercicio anterior para el caso de matrices simétricas, y compruebe que $A=LL^t$ y que $A = LDL^t$. b) Use los comandos `triu()` (o `tril()`) para crear matrices triangulares y multiplique por su transpuesta, al resultado descompongalo con `scipy.linalg.cholesky()` ¿qué concluye? 

**Tarea 6)**: Una matriz a banda $A$ de $n\times n$ es aquella que cumple que todos sus elementos son cero fuera de una zona diagonal cuyo rango se determina por el número de diagonales inferiores $l$ y superiores $u$ distintas de cero, osea si $a_{ij}$ son los elementos de la matriz, entonces

 $$a_{ij}=0\quad {\mbox{si}}\quad j<i-l\quad {\mbox{ o }}\quad j>i+u;\quad l,u\geq 0.$$

a) Escribir un código python que crea matrices a bandas de forma general para valores dados $n,l,u$.
b) Modificar el código para estraer las $l$ diagonales inferiores y las $u$ diagolales superiores de una matriz dada $A$ y almacenar los elementos distintos de cero en una matriz $B$ de dimensiones $(k,n)$ con $k=l+u+1$ y componentes $b_{ij}$ de tal forma que se pueda usar en `scipy.linalg.solve_banded((l,u),B,d)` para resolver $Ax=d$. Por ejemplo, si $A$ es $6\times 6$ con $u =1$, $l =2$ entonces $B$ debe tener la forma
$$
B=
\begin{pmatrix}
 *    &  a_{01} & a_{12}&  a_{23}&  a_{34}&  a_{45}\\
a_{00}&  a_{11} & a_{22}&  a_{33}&  a_{44}&  a_{55}\\
a_{10}&  a_{21} & a_{32}&  a_{43}&  a_{54}&       *\\
a_{20}&  a_{31} & a_{42}&  a_{53}&      * &       *\\
\end{pmatrix}
$$

Ayuda: considere el hecho de que 

$$b_{ij} = a_{jq} \quad\hbox{para}\quad i = 0,...,(k-1) \quad\hbox{y}\quad  j = 0,...,(n-1), $$

donde $q = i+j-l.$

**Tarea 7)** Genere una matriz definida positiva y resuelva el sistema $A\mathbf{x}=\mathbf{b}$, compare los tiempos de computo para cada uno de los casos $LU$, $LL^t$ y $LDL^t$ respecto al tiempo dado por el comando `solve(A,b)` (para que el tiempo sea significativo, para cada caso haga un bucle que repita la operación unas 500 veces, haga un histograma de los tiempos usando `plt.his()`).

**Tarea 8)** Cuando una matriz tridiagonal tiene la forma, 
$$
A=\begin{pmatrix}a & b\\
c & a & b\\
 & \ddots & a & \ddots \\
 &  &   & \ddots &  \\
 &  &  & c & a
\end{pmatrix},
$$
se conoce como *matriz Toeplitz* y [se puede demostrar](https://doi.org/10.1016/S0024-3795(99)00114-7) que los autovalores son dados por

$$\lambda_{k}=a+2\sqrt{bc}\cos\left[\frac{k\pi}{(n+1)}\right], \quad k=1\cdots n$$

Haga un programa que calcule el error relativo en los autovalores para una matriz de $n\times n$ usando `LA.eig(A)`
(Use $n=10, a=5, b=2$ y $c=3).$

**Tarea 9)** En física muchas veces solo necesitamos encontrar el autovalor más grande y el autovalor más pequeño de una matriz.

  a) Genere una matriz $A$ de dimensión (10,10) y verifique que es diagonalizable.

  b) Para Encontrar el autovector asociado al autovalor más grande genere un vector $\mathbf{u}$ aleatorio y normalicelo $\mathbf{x} = \mathbf{u}/\|\mathbf{u}\|$, luego usando el siguiente procedimiento iterativo (use como condición de parada el error relativo, $e=\frac{\|\mathbf{x}_{n+1}-\mathbf{x}_n\|}{\|\mathbf{x}_{n+1}\|}$), 

$$
\mathbf{u}_{n+1} = A\mathbf{x}_n,
$$ 

muestre que $\|\mathbf{u}_n\|$ converge rápido al autovector más grande de $A$, calcule el autovector como,

$$\lambda_{max}=\frac{\mathbf{x}_n\cdot A\mathbf{x}_n}{\mathbf{x}_n\cdot \mathbf{x}_n}.$$
  
  c) Para encontrar el autovalor más pequeño repita el mismo procedimiento pero use la inversa de $A$, es decir 
  
$$
  \mathbf{u}_{n+1} = A^{-1}\mathbf{x}_n\\
  \lambda_{min}=\frac{\mathbf{x}_n\cdot A\mathbf{x}_n}{\mathbf{x}_n\cdot \mathbf{x}_n}
$$
  
$1/\|\mathbf{u}_n\|$ converge rápidamente a el autovector más pequeño en valor absoluto (esto es debido a que $1/\lambda$ converge al autovalor más grande de $A^{-1}$, entonces $\lambda$ converge al autovalor más pequeño de $A$).  
 
 Finalmente verifique la respuesta diagonalizando con python la matriz.

**Tarea 10)**: La técnica del *cociente de Rayleigh* sirve para dar una aproximación de un autovector con su respectivo autovalor si se da un aproximación inicial $b_{0}$ del autovector y tiene convergencia cúbica, la iteración se define por, 

$$
\mathbf{b}_{i+1}={\frac {(A-\lambda _{i}I)^{-1}\mathbf{b}_{i}}{\|(A-\lambda_{i}I)^{-1}\mathbf{b}_{i}\|}},$$

donde,

$$
\lambda_{i+1}={\frac {\mathbf{b}_{i+1}\cdot A\mathbf{b}_{i+1}}{\mathbf{b}_{i+1}\cdot \mathbf{b}_{i+1}}}.
$$

Implemente el código python mediante la rutina de gauss jordan para el cálculo de la inversa, use como condición de parada el error relativo, $e=\frac{\|\mathbf{b}_{i+1}-\mathbf{b}_i\|}{\|\mathbf{b}_{i+1}\|}$.


<a id='problema10'></a> 
**Tarea 11)**: la descomposición $LU$ de una matriz tridiagonal es dada por,
 
$$
\begin{equation*}
\begin{pmatrix}
   {b_1} & {c_1} & {   }  & {   }  & { 0 } \\
   {a_2} & {b_2} & {c_2}  & {   }  & {   } \\
   {   } & {a_3} & {b_3}  & \ddots & {   } \\
   {   } & {   } & \ddots & \ddots & {c_{n-1}}\\
   { 0 } & {   } & {   }  & {a_n}  & {b_n} \\
\end{pmatrix}
= 
\begin{pmatrix}
   1     &       & {   }  & {   }  & { 0 } \\
   {l_2} & 1     &        & {   }  & {   } \\
   {   } & {l_3} & 1      &        & {   } \\
   {   } & {   } & \ddots & \ddots & {   } \\
   { 0 } & {   } & {   }  & {l_n}  & 1 \\
\end{pmatrix}
\begin{pmatrix}
   {v_1} & {c_1} & {   }  & {   }  & { 0 } \\
   {   } & {v_2} & {c_2}  & {   }  & {   } \\
   {   } & {   } & {   }  & \ddots & {   } \\
   {   } & {   } &        & \ddots & {c_{n-1}}\\
   { 0 } & {   } & {   }  & {   }  & {v_n} \\
\end{pmatrix}
\end{equation*}
$$ 

donde,

$$
\begin{eqnarray}
v_1 &=& b_1\\
l_k &=& a_k/v_{k−1}\\
v_k &=& b_k−l_kc_{k−1} \quad \hbox{con}\quad  k= 2, . . . , n
\end{eqnarray}
$$

Ahora si se quiere resolver el sistema, $A\mathbf{x}=\mathbf{d}$, se resuelve, $L\mathbf{y}=\mathbf{d}$, como,

$$
\begin{eqnarray}
x_n &=& y_n/v_n\\
y_k &=& d_k−l_ky_{k−1}, \quad \hbox{con}\quad  k= 2, . . . , n
\end{eqnarray}
$$

y se resuelve, $U\mathbf{x}=\mathbf{y}$ como,

$$
\begin{eqnarray}
x_n &=& y_n/v_n\\
x_k &=& (y_k−c_kx_{k+1})/v_k, \quad \hbox{con}\quad  k=n−1, . . . ,1
\end{eqnarray}
$$

Implemente el código python que calcula $L,U$ y resuelve $A\mathbf{x}=\mathbf{d}$.

**Tarea 12)**: La integral de Fredholm de segunda clase es ampliamente utilizada en física, por ejemplo, en procesamiento de señales y en radiología en fenómenos de transporte de radiación, se define como,

$$u(x)=f(x)+\int _{a}^{b}K(x,t)u (t)\,dt,$$

donde el kernel $K(x,t)$ y $a,b$ son dados, para aproximar la función  $u$ se hace la partición $a=x_0<x_1<...<x_m=b$, que se crea la variables $u_i=u(x_i)$ y da el sistema de $m$ ecuaciones,

$$u(x_i)=f(x_i)+\int _{a}^{b}K(x_i,t)u (t)\,dt \quad \hbox{con}\quad  i=0, . . . ,m.$$ 

Si se toma $a = 0, b = 1, f (x) = x^2$, y $K(x, t) = e^{|x−t|}$: 

a) muestre que el sistema lineal:

$$
\begin{eqnarray}
u(0) = f (0) + \frac{1}{2}[K(0, 0)u(0) + K(0, 1)u(1)],\\
u(1) = f (1) + \frac{1}{2}[K(1, 0)u(0) + K(1, 1)u(1)],
\end{eqnarray}
$$

se puede resolver con regla trapezoidal, encuentre $u_0,u_1$.<br>
b) Implemente la regla trapezoidal y de Simpson para cualquier $f,K,m$ en $[a,b]$ (use las librerías de `scipy.linalg` para eliminación gaussiana y también con descomposición $LU$) ¿Se puede resolver con cuadratura gaussiana?<br>
c) Grafique $u(x)$ y $u'(x)$ en $[a,b]$ con pasos de $10^{-1}$ y $10^{-4}$. <br>
d) ¿Cuándo es aconsejable hacer interpolación con splines cúbicos? resolver el sistema para pasos de $10^{-1}$ hacer interpolación y comparar a la solución de $10^{-4}$.<br>


**Tarea 13)**: Una matriz antisimétrica se define como,

$$\Omega = \frac{A-A^t}{2}$$

se puede demostrar que en tres dimensiones esto es equivalente a la matriz que genera el producto cruz,

$$
\Omega=[\omega ]_{\times }={\begin{bmatrix}\,\,0&\!-\omega _{3}&\,\,\,\omega _{2}\\\,\,\,\omega _{3}&0&\!-\omega _{1}\\\!-\omega _{2}&\,\,\omega _{1}&\,\,0\end{bmatrix}}.
$$

En python hacer una rutina que genere la velocidad angular a partir de matrices aleatorias $A$, calcule la velocidad tangencial $\mathbf{v}_{\perp}$ y verifique que se cumple la siguiente expresión,

$$\mathbf{v}_{\perp}=\Omega\cdot\mathbf{r}=\boldsymbol{\omega} \times\mathbf{r},$$

para un vector $\mathbf{r}$ aleatorio cualquiera (ver sección [Efecto de multiplicar una matriz por un vector](Clases_17_18_Algebra_Lineal.ipynb#Efecto_matriz_por_un_vector)).

**Tarea 14)**: El siguiente circuito eléctrico se describe por las ecuaciones
$$
\begin{matrix}
-V_1 +R_1I_1+R_2(I_1-I_2) = 0.0 \\
R_2(I_2-I_1)+R_3I_2+R_4(I_2-I_3)=0.0 \\
R_4(I_3-I_2)+R_5I_3+V_2= 0.0 
\end{matrix}
$$

a) Encuentre las corrientes $I_1, I_2, I_3$ si $R_1=1.1$ K$\Omega$, $R_2=2.3$ K$\Omega$, $R_3 = 1.5$ K$\Omega$, $R_4 = 0.55$ K$\Omega$, $R_5 = 1.6$ K$\Omega$, $V_1 = 20$ V y $V_2=15$ V.
b) Grafique $I_1, I_2, I_3$ como función de $V_1$ en el intervalo $[5,30]$ V.

**Tarea 15)**: Sobre una barra estática de longitud 7.80m actuan 4 fuerzas, en un extremo $F_0=926$N con ángulo de $90^\circ$, $F_1$  se aplica en el otro extremo con ángulo de $69.3^\circ$, $F_2$ se aplica a una distancia de $1.50$m del primer extremo con ángulo de $251.1^\circ$, y $F_3$ se aplica distancia de $2.60$m del otro extremo con ángulo de $303.4^\circ$.

a) Hacer el diagrama de fuerzas y demuestre que la suma de fuerzas y torques da las ecuaciones:<br>

Suma de fuerzas verticales:
$$F_1 \text{sen}\,69.3° − F_2\text{sen}\,71.1° − F_3 \text{sen}\,56.6° + 926 = 0,$$

suma de fuerzas horizontales:
$$F_1 \cos 69.3° − F_2 \cos 71.1° + F_3 \cos 56.6° = 0,$$

torques:
$$7.80F_1\text{sen}\,69.3° − 1.50F_2 \text{sen}\,71.1° − 5.20 F_3\text{sen}\,56.6° = 0.$$

b) Use python calcular las fuerzas $F_1, F_2, F_3$. <br>
c) Variar el ángulo de $F_0$ de $0$ a $180^\circ$ y graficar $F_1, F_2, F_3$ como función del ángulo.<br>
d) Agrege tres fuerzas más con diferentes ángulos y a puntos diferentes y repita los pasos anteriores. 

In [ ]:
# Ayuda numeral a)
# Representación artística de fuerzas sobre una barra (escala no real)
# Hacer flecha con arrow(x,y, dx,dy, **kwargs), comienza en (x, y) hasta (x+dx, y+dy).

import matplotlib.pyplot as plt
from numpy import *

θ1=69.3*pi/180; θ2=71.1*pi/180; θ3=56.6*pi/180 
F = .1 # factor de escala para flechas (fuerzas).

plt.figure(figsize=(15, 3))
plt.xlim(-.2,1.2)
plt.ylim(-.2,.2)
plt.axis('off') # remover axis, (lineas con números en ejes  x, y)

# Barra 
plt.hlines(0,-.1,1.1,color="k",lw=1) # sistema de referencia
plt.hlines(0,1,0,color="chocolate",lw=10) # Barra
# Fuerza F1  
plt.arrow(0, 0, 0, F, head_width=0.02, head_length=.02, fc='k', ec='k',lw=5)
plt.text(.0, 0.15, "926 N")
# Fuerza F2
plt.arrow(1,0, F*cos(θ1), F*sin(θ1), head_width=0.02, head_length=.02, fc='k', ec='k',lw=5)
plt.text(1.02, 0.01, r"69.3$^o\quad\bf F_1$")
# Fuerza F3
plt.arrow(0.3, 0,-F*cos(θ2), -F*sin(θ2), head_width=0.02, head_length=.02, fc='k', ec='k',lw=5)
plt.text(0.18, -0.05, r"71.1$^o\quad\bf F_2$")

plt.arrow(0.7, 0,F*cos(θ3), -F*sin(θ3), head_width=0.02, head_length=.02, fc='k', ec='k',lw=5)
plt.text(.75, -0.05, r"56.6$^o\quad\bf F_3$")

# Distancias torques
plt.text(.15, 0.02, "1.50 m")
plt.text(.45, 0.02, "3.70 m")
plt.text(.85, 0.02, "2.60 m")
180-251.1, 360.0-303.4

**Tarea 16)**: Sea $B=A^{-1}$ la inversa de una matriz triangular $A$, si la inversa $B$ se puede calcular con el siguiente algoritmo
```python
n = dimensión de A;
B = zeros;
for i=1:n
    B(i,i) = 1/A(i,i);
    for j=1:i-1
        s = 0;
        for k=j:i-1
            s = s + A(i,k)*B(k,j);
        end
        B(i,j) = -s*B(i,i);
    end
end
```
a) Implemente el código en python.<br>
b) Una de las maneras más efectivas de calcular la inversa de una matrix simétrica (o hermítica) es por descomposición de Cholesky, $A=LL^\dagger$, use la rutina anterior para calcular la inversa. Resuelva por descomposición $PA=LU$ y por Gauss-Jordan y compare los tiempos de ejección (tenga en cuenta que los tiempos dependeran de si las rutinas usadas son python puro o son de scypy).<br>
c) Calcular el determinate de la  inversa (Ayuda: note que $\det(A) = \prod_{i=1}^{n}l_{ii}^{2}$ por lo que $\det(B) = \prod_{i=1}^{n}l_{ii}^{-2}$).

<a id='Material_complementario'></a> 
# Material complementario
Algunas definiciones, métodos y rutinas adicionales.

El siguiente algoritmo también calcula la descomposición $LDL^t$, pero en $L$ la diagonal tiene entradas diferentes de uno:

In [ ]:
def ldl(A):
    A = np.asmatrix(A) # si es array, ver como matriz 
    # verificar que A es simetrica o hermitica
    if np.allclose(A.H, A): # (A.H == A).all()
        S = np.diag(np.diag(A)) # matriz con los elementos de la diagonal de A
        Sinv = np.diag(1/np.diag(A)) # inversa de S
        D = np.matrix(S.dot(S))      # D = S^2
        
        Lch = np.linalg.cholesky(A)  
        L = np.matrix(Lch.dot(Sinv)) # L = Lch*S^-1 
        return L, D
    else:
        print("A debe ser Simétrica or Hermítica.")
        return None, None

    
M = np.array([[7, 3, -1, 2], 
              [3, 8, 1, -4], 
              [-1, 1, 4, -1], 
              [2, -4, -1, 6] ])
 
L, D = ldl(M)
print("L =\n",L)
print("\nD = \n",D)

# recuperar matriz original:
print ("\nLDL^t =\n", L*D*L.H)

### Producto exterior
Sean $\mathbf {u} =\left(u_{1},u_{2},\dots ,u_{m}\right)$ y $\mathbf {v} =\left(v_{1},v_{2},\dots ,v_{n}\right)$ dos vectores cualquiera de dimensión $m$ y $n$, se define el producto exterior (*outer product*) como $\mathbf {u} \mathbf {v}^t$, que da la matriz $m\times n$ obtenida de hacer el producto,

$$
\mathbf {u} \mathbf {v} ^t
={\begin{pmatrix}u_{1}\\u_{2}\\ \vdots\\u_{m}\end{pmatrix}}{\begin{pmatrix}v_{1}&v_{2}&\cdots&v_{n}\end{pmatrix}} ={\begin{pmatrix}u_{1}v_{1}&u_{1}v_{2}&\dots &u_{1}v_{n}\\u_{2}v_{1}&u_{2}v_{2}&\dots &u_{2}v_{n}\\\vdots &\vdots &\ddots &\vdots \\u_{m}v_{1}&u_{m}v_{2}&\dots &u_{m}v_{n}\end{pmatrix}}
$$

note que el producto esterior es diferente del producto punto, $\mathbf {u}^t\mathbf {v}$, en python se calcula con el método numpy,
```python
np.outer(u, v, out=None)
```
o también,
```python
np.c_[u]*v # c_[u] crea vector columna
```
Si $\mathbf {u}$ y $\mathbf {v}$ son complejos entonces hay que tomar primero el conjugado de $\mathbf {v}^\dagger$, `np.outer(u, v.conj())`.
Ejemplo:

In [ ]:
# Producto exterior
u = np.array([1,2,3])
v = np.array([2,2,2])
np.outer(u,v), np.outer(v,u) # outer no conmuta

### Transformación de Householder
Sea $\mathbf {u}$ un vector unitario cualquiera, la transformación de Householder define por,

$$P=I -2\mathbf {u}\mathbf {u}^t$$

note que  $\mathbf {u}\mathbf {u}^t$ es una matriz dada por el producto exterior,
es fácil demostar (hacerlo) que la matriz $P$ es simétrica y ortogonal, $P=P^t=P^{-1}$.

Ejemplo:

In [ ]:
# Householder
u = np.random.rand(3)
u = u/np.dot(u,u)**.5
P = np.eye(3)-2.*np.outer(u,u)
# Note que P.T == P, y P@P.T == I ósea P.T == inv(P)
P, P.T, P@P.T 

<a id='Descomposición_QR'></a> 
### Descomposición QR 

La descomposición QR de una matriz se define como el producto de una matriz ortogonal $Q$ (unitaria si $A$ es compleja) por una triangular superior $R$, ósea $A= QR$, esta se puede implementar por varios métodos, mediante el método de ortogonalización de Gram-Schmidt, rotaciones de Givens y por reflexiones de Householder.

#### Método Schwarz-Rutishauser
El algoritmo de Schwarz-Rutishauser es una modificación del clásico proceso de ortogonalización de Gram-Schmidt, propuesto por H. R. Schwarz, H.

#### Método de Householder
El método de Householder o reflexión de Householder es una transformación que refleja el espacio con respecto a un plano determinado, el procedimiento es: 

Sea ${\mathbf {x}}=(x_1,x_2,...x_n)$ el primer vector columna tomado de la matriz $A$ tal que su magnitud sea ${\|\mathbf {x}}\| = |α|$ (donde $\alpha$ es un escalar), si $A$ es real, $\alpha$ debe adoptar el signo contrario de la primera componente $x_1$ de ${\mathbf {x}}$ para evitar pérdida de precisión, pero en complejos se generaliza como,

$$\alpha =-e^{i\arg x_{1}}\|\mathbf {x} \|$$


luego se define  el vector $\mathbf {e} _{1}=(1 0 … 0)^t$ y se hace la operación,

$$
\begin{aligned}
\mathbf {u} &=\mathbf {x} -\alpha \mathbf {e} _{1},\\
\mathbf {v} &={\mathbf {u} \over \|\mathbf {u} \|},\\
Q&=I-2\mathbf {v} \mathbf {v} ^\dagger.
\end{aligned}
$$

Luego se considera el producto de $QA$ que da la matriz,

$$
R_1\equiv Q_{1}A={\begin{pmatrix}\alpha _{1}&* &\dots &* \\0&&&\\\vdots &&A_1'&\\0&&&\end{pmatrix}},
$$

donde se redefine $Q=Q_1$, este proceso se puede repetir gradualmente para la submatriz $A_1′$ obtenida de borrar de $R_1$ su primera fila y columna y se repiten las operaciones anteriores a partir de la primera columna de $A_1'$ para obtener la submatriz de Householder $Q′_2$ (Note que $Q′_2$ tiene una dimensión menos que $Q_1$), en general después de $k$ repeticiones de las operaciones se obtiene la matriz $n\times n$,

$$
Q_{k}={\begin{pmatrix}I_{k-1}&0\\0&Q_{k}'\end{pmatrix}},
$$

donde $I_{k-1}$ es la matriz identidad de dimensión $k-1$ y $Q'_k$ es la submatriz reducida de Householder de dimensión $(n-k)\times (n-k)$ obtenida en el procedimiento al paso $k$. Finalmente, después de $n-1$ iteraciones de este proceso, se obtiene la matriz triangular superior,

$$
R=Q_{n}\cdots Q_{2}Q_{1}A$$

y la matriz unitaria,

$$
Q=Q_{1}^\dagger Q_{2}^\dagger\cdots Q_{n}^\dagger,
$$

con la propiedad buscada de $A = QR$ como descomposición de $A$, pues de la definición de Householder recordar que $Q_k^\dagger=Q_k$ y que $Q_k^\dagger Q_k=I$, por lo que se recupera $A$, en scipy se implementa en la rutina,
```python 
scipy.linalg.qr()
```
(para más [ver wikipedia](https://en.wikipedia.org/wiki/QR_decomposition)). Veamos la implemetación:

In [ ]:
# Con los reales coinciden los resultados a LA.rq() con complejos
# difiere, pero Q es ortogonal y R es triu.
n = 100
A = np.random.randn(n,n) + 1j*np.random.randn(n,n)

def QR(A):
    n = len(A)
    Ak = np.copy(A)
    Q = np.eye(n,n, dtype=complex) 
    
    for k in range(n-1):
        x = Ak[0:,0]  # Columna 1 de Ak, dimensión n-k.
        x[0] += np.exp(1j*np.angle(x[0]))*LA.norm(x) # α = -exp(-i arg(x0))|x|
        # Householder transformación 
        Qp = np.eye(n-k,n-k) - 2.*np.outer(x,x.conj())/LA.norm(x)**2.
        Qk = np.eye(n,n, dtype=complex) # Matriz Q al paso k.
        Qk[k:,k:] = Qp# Matriz reducida Q' al paso k. 
        Q = Q@Qk      # Recordar que Qk.T == Qk, es hermítica.
        Ak = (Qp@Ak)[1:,1:] # A′ se obtiene de Q1A al eliminar la primera fila y columna.
        #print('Qk=\n',np.round(Qk,3))
        #print('Ak=\n',np.round(Ak,3))
        #print('Q=\n',np.round(Q,3))
    R = Q.T.conj()@A 
    return Q, R    


def QR1(A): # un poco más lenta en matrices grandes.
    n = len(A)
    R = np.copy(A)
    Q = np.eye(n,n, dtype=complex) 
    
    for k in range(n-1):
        x = np.copy(R[k:,k])   # Columna 1 de Ak, dimensión n-k.
        x[0] += np.exp(1j*np.angle(x[0]))*LA.norm(x) # α = -exp(-i arg(x0))|x|
        # Householder transformación 
        Qp = np.eye(n-k,n-k) - 2.*np.outer(x,x.conj())/LA.norm(x)**2.
        Qk = np.eye(n,n, dtype=complex) # matriz Q al paso k.
        Qk[k:,k:] = Qp  # matriz reducida Q' al paso k.                 
        Q = Q@Qk        # recordar que Qk.T == Qk, es hermítica.
        R = Qk@R        # matriz A a paso k.
    return Q, R    


# Método Schwarz-Rutishauser Algoritmo
def QR2(A, type=complex): 
 
     A = np.array(A, dtype=type)
     m,n = np.shape(A)
 
     R = np.zeros((n, n), dtype=type)
     Q = np.array(A, dtype=type)

     for k in range(n):
         for i in range(k):
             R[i,k] = np.transpose(Q[:,i]).dot(Q[:,k])
             Q[:,k] = Q[:,k] - R[i,k] * Q[:,i]
 
         R[k,k] = LA.norm(Q[:,k])
         Q[:,k] = Q[:,k]/R[k,k]
 
     return -Q, -R


Q, R = QR(A)
#Q, R = LA.qr(A)
I = Q@Q.T.conj()
#np.round(R,3),np.round(Q,3),np.round(I,3), LA.det(Q) # verificación
(abs(Q@R - A) < 1e-13).all() # verificar que QR = A

In [ ]:
%timeit QR1(A)
%timeit QR(A)

### Método de diagonalización QR
Antes del descubrimiento del [algoritmo QR](https://en.wikipedia.org/wiki/QR_algorithm), para diagonalizar había que calcular los ceros de la ecuación característica, 
$$f(\lambda)=\hbox{det}(A - \lambda I)=0,$$
el problema es que para $n$ grande este método es inestable en aritmética de punto flotante, pero en 1959 Francis descubrió el método QR para diagonalización, en su forma más simple este es: sea $A$ una matriz simétrica tal que se descompone como el producto de una una matriz ortogonal $Q$ por una triagular superior $R$, entonces se produce la serie,

$$
A_{k+1}=R_{k}Q_{k}=Q_{k}^{T}Q_{k}R_{k}Q_{k}=Q_{k}^{T}A_{k}Q_{k}=Q_{k}^{-1}A_{k}Q_{k}
$$

luego todas las $A_k$ son matrices semejantes lo que implica que tienen los mismos valores propios. El algoritmo es numéricamente estable pues opera por transformaciones ortogonales, para la construción del algoritmo se toma $A_0=A$, y se aplica descomposición QR, el seudocódigo es,

\begin{aligned} 
&\text{$A_0 = A$,}\\ 
&\text{for k = 1,2, ... }\\
&\quad\text{$A_{k-1} = Q_kR_k,\quad$ aplicar descomposición QR a $A_{k-1}$,}\\
&\quad\text{$A_k = R_kQ_k,\,\,\quad\,\,$ su diagonal converge a los autovalores,}\\
&\quad\text{$Q = Q_1,...,Q_k,$ sus columnas convergen a los autovectores,}\\
&\text{end} 
\end{aligned}

Como condición de parada se puede usar $|\max(A_k-A_{k-1})|<\epsilon$ o también $|\sum_{i,j} ((a_{ij})_k-(a_{ij})_{k-1})|<\epsilon$, 
veamos la implementación en python:

In [ ]:
# Versión más simple: retorna autovalores y autovectores.
A = np.random.rand(4,4)
A =  (A+A.T)*.5 # A debe ser simétrica (hermítica).
steps = 100     # Si Ak no da diagonal incremente steps.

Ak= np.copy(A) 
Q = np.eye(len(Ak))
for i in range(steps): 
    Qk,R = LA.qr(Ak) # QR(Ak) 
    #print(np.round(Ak,3),'\n') 
    Ak = R@Qk   # Ak debe terner los mismos autovalores de A.
    Q  = Q@Qk   # Q es el producto de las Qk.        

np.round(Ak,3), np.round(Q,3), LA.eig(A) # Ak da autovalores, Q da autovectores.

In [ ]:
# Código completo: EIGH(A) retorna autovalores y autovectores.
import numpy as np

def QR(A):
    n = len(A)
    Ak = np.copy(A)
    Q = np.eye(n,n, dtype=complex) 
    
    for k in range(n-1):
        x = Ak[0:,0]  # Columna 1 de Ak, dimensión n-k.
        x[0] += np.exp(1j*np.angle(x[0]))*np.linalg.norm(x) # α = -exp(-i arg(x0))|x|
        # Householder transformación 
        Qp = np.eye(n-k,n-k) - 2.*np.outer(x,x.conj())/np.linalg.norm(x)**2.
        Qk = np.eye(n,n, dtype=complex) # Matriz Q al paso k.
        Qk[k:,k:] = Qp# Matriz reducida Q' al paso k. 
        Q = Q@Qk      # Recordar que Qk.T == Qk, es hermítica.
        Ak = (Qp@Ak)[1:,1:] # A′ se obtiene de Q1A al eliminar la primera fila y columna.
    
    R = Q.T.conj()@A 
    return Q, R    

def EIGH(A, eps=1e-14, steps=None):
    n = len(A)
    if steps==None: steps = n**4
    
    Ai = np.copy(A) 
    Q  = np.eye(n)
    for i in range(steps): 
        Qk,R = QR(Ai) 
        
        Ak = R@Qk   # Ak debe terner los mismos autovalores de A.
        Q  = Q@Qk   # Q es el producto de las Qk.
        
        if np.abs((Ak - Ai).sum())<eps: break # Condición de parada.
        Ai = Ak
    print("Steps:", i)    
    return np.diag(np.round(Ak,8)), np.round(Q,8) # Ak da autovalores, Q da autovectores.

# Test
n = 3
A = np.random.rand(n,n) # + 1j*np.random.rand(n,n)
A =  (A+A.T.conj())*.5 # A debe ser simétrica (hermítica).
EIGH(A), np.linalg.eig(A)

### Como hacer matrices aleatorias que sean ortogonales o unitarias
El método Householder sirve para crear una matriz ortogonal, veamos la implementación de una matriz ortogonal:

In [ ]:
# Crear matriz ortogonal con Householder:
def rvs(n=3): # Matriz ortogonal
    Q = np.eye(n,n) 
    
    for k in range(n-1):
        x = np.random.randn(n-k)
        x[0] += (np.exp(1j*np.angle(x[0]))*LA.norm(x)).real # α = -exp(-i arg(x0))|x|
        # Householder transformación 
        Qp = np.eye(n-k,n-k) - 2.*np.outer(x,x.conj())/LA.norm(x)**2.
        Qk = np.eye(n,n) # Matriz reducida Q' al paso k.
        Qk[k:,k:] = Qp   # Matriz Q al paso k.
        Q = Q@Qk         # Recordar que Qk.T == Qk, es hermítica.
    return Q    

A = rvs(3)   
LA.det(A), A@A.T  # Da matriz identidad y det

Estos métodos ya están implementados en scipy, veamos: 

In [ ]:
# Crear matriz unitaria con Householder:
def rvsu(n=3): # matriz unitaria
    Q = np.eye(n,n, dtype=complex) 
    
    for k in range(n-1):
        x = np.random.randn(n-k) + 1j*np.random.randn(n-k)
        x[0] += np.exp(1j*np.angle(x[0]))*LA.norm(x) # α = -exp(-i arg(x0))|x|
        # Householder transformación 
        Qp = np.eye(n-k,n-k) - 2.*np.outer(x,x.conj())/LA.norm(x)**2.
        Qk = np.eye(n,n, dtype=complex) # Matriz reducida Q' al paso k.
        Qk[k:,k:] = Qp                  # Matriz Q al paso k.
        Q = Q@Qk      # Recordar que Qk.T == Qk, es hermítica.
    return Q    
    
U=rvsu(3)
abs(LA.det(U)), np.round(U@U.T.conj(),3)

In [ ]:
# Crear matrices ortogonales y unitarias con scipy: 
from scipy.stats import ortho_group 
from scipy.stats import unitary_group

A = ortho_group.rvs(3)   # Matriz ortogonal.
U = unitary_group.rvs(3) # Matriz unitaria.
A@A.T, U@U.conj().T      # Verificación.

### Crear matrices aleatorias con autovalores predeterminados
Primero se crea una matriz aleatoria $S$ con autovalores predeterminados $D=\,$diag$\{\lambda_1,\lambda_2,...,\lambda_n\}$, se hace la operación, $S=ADA^{-1}$, donde $A$ es una matriz aleatoria cualquiera. Si se requiere que $S$ sea simétrica entonces $A$ debe ser ortogonal (o unitaria si se requiere que $S$ sea hermítica), en ese caso se usa `svr(n)` para crear $A$ (esta rutina existe en scipy), en general $S$ tendrá autovalores predefinidos por $D$.

In [ ]:
# Crear matriz aleatoria con autovalores predefinidos,
#     D = diag{λ1,λ2,...,λn}:

A = np.random.rand(4,4) # Si A es aleatoria entonces,
D = np.diag([2,4,5,6])  # Crear matriz diagonal, luego.
S = A@D@LA.inv(A)       # Matriz aleatoria con autovalores dados. 
LA.eig(S)               # Por D, veamos: 

In [ ]:
# Crear matriz simétrica aleatoria con autovalores predefinidos,
#    D = diag{λ1,λ2,...,λn}:

A = rvs(4)          # Para que S sea simétrica A debe ser ortogonal.
D = np.diag([2,4,5,6])  # Crear matriz diagonal, con valores deseados.
S = A@D@A.T             # Matriz aleatoria con autovalores deseados.
LA.eig(S), S            # Verificar. 

In [ ]:
# Crear matriz hermítica aleatoria con autovalores predefinidos,
#    D = diag{λ1,λ2,...,λn}:

U = rvsu(4)              # U debe ser unitaria.
D = np.diag([2,4,6,8])   # Autovalores deseados.
S = U@D@U.T.conj()       # Matriz hermítica con autovalores deseados.
LA.eig(S), np.round(S,3) # Verificar. 

In [ ]:
# Otra versión: Crear matriz ortogonal con Householder.

# https://stackoverflow.com/questions/38426349/how-to-create-random-orthonormal-matrix-in-python-numpy
import numpy as np    

def rvs(dim=3):
    H = np.eye(dim)
    D = np.ones((dim,))
    for n in range(1, dim):
        x = np.random.randn(dim-n+1)
        D[n-1] = np.sign(x[0])
        x[0] -= D[n-1]*np.sqrt((x*x).sum())
        # Householder transformation
        Hx = np.eye(dim-n+1) - 2.*np.outer(x, x)/(x*x).sum()
        mat = np.eye(dim)
        mat[n-1:, n-1:] = Hx
        H = np.dot(H, mat)
        # Fix the last sign such that the determinant is 1
    D[-1] = (-1)**(1-(dim % 2))*D.prod()
    # Equivalent to np.dot(np.diag(D), H) but faster, apparently
    H = (D*H.T).T
    return H

A = rvs(dim=3)   
LA.det(A), A@A.T,  # Da matriz identidad y det

In [ ]:
# Reprecentación artística de la rotación de los ejes en 2D
# para que coincidan con los ejes de rotación de una barra.

θ = 30*pi/180 #  Ángulo en grados a radianes
r = 1  # factor de escala.

# ---------------------------------------------------------------
import matplotlib.pyplot as plt
from numpy import *
from matplotlib import patches


f, ax = plt.subplots(figsize=(10, 10))
#plt.figure(figsize=(10, 10))
plt.xlim(-1.1*r/2.1, 1.1*r)
plt.ylim(-1.1*r/2.2, 1.1*r)
plt.axis('off') # remover axis, (lineas con números en ejes  x, y)


def R(θ): # Matriz de rotación por ángulo θ 
    return  r*array([[cos(θ), sin(θ)],
                     [-sin(θ), cos(θ)]])

# 1) Dibujar barra
A = R(θ)
plt.arrow(-A[0,0]/2,-A[0,1]/2,A[0,0],A[0,1], head_width=0.,  head_length=.0, fc='c', ec="chocolate",lw=15)

# 2) Dibujar arco de rotación 
plt.annotate(r'$\bfω$', xy=(.48, 0.34), fontsize='xx-large')
patch = patches.Arc(xy=(.5, 0.3), width=0.2, height=0.1, angle=120, theta1=0, theta2=280)
plt.arrow(.45 ,0.385, 0.001, 0.0001, head_width=0.02,  head_length=.02, fc='k', ec='k',lw=1)
ax.add_patch(patch)

# 3) Dibujar arco ángulo
a = linspace(0, θ, 20)
xa = 0.3*r*cos(a)
ya = 0.3*r*sin(a)
plt.plot(xa, ya, c='r', lw=3)
plt.annotate('θ', xy=(.2, 0.02), fontsize='xx-large')


# 4) Sistema rotado de ejes
plt.arrow(0, 0, A[0,0],A[0,1], head_width=0.02,  head_length=.02, fc='b', ec='b',lw=2)
plt.arrow(0, 0, A[1,0],A[1,1], head_width=0.02,  head_length=.02, fc='b', ec='b',lw=2)
# Nombres de los ejes
A = R(θ - 4*pi/180)
plt.text(  .9*A[0,0], .9*A[0,1], r"$x'$", fontsize='xx-large', c='b')
plt.text( 1.2*A[1,0], .8*A[1,1], r"$y'$", fontsize='xx-large', c='b')

# 5) Sistema original de ejes
A = R(0)
plt.arrow(0, 0, A[0,0],A[0,1], head_width=0.02, head_length=.02, fc='k', ec='k',lw=2)
plt.arrow(0, 0, A[1,0],A[1,1], head_width=0.02,  head_length=.02, fc='k', ec='k',lw=2)
# Nombres de los ejes
plt.text( .9,       -.1,       r"$x$", fontsize='xx-large', c='k')
plt.text(-.1,        .9,       r"$y$", fontsize='xx-large', c='k')

plt.savefig("../figures/Rotacion_ejes.png",transparent=True,format='png')


# Referencias:
https://mathoverflow.net/questions/131527/eigenvalues-of-symmetric-tridiagonal-matrices

calcular matriz tridiagonal
https://www.webpages.uidaho.edu/~barannyk/Teaching/LU_factorization_tridiagonal.pdf

How to generate a random unitary matrix:
http://home.lu.lv/~sd20008/papers/essays/Random%20unitary%20[paper].pdf

The QR Algorithm
https://people.inf.ethz.ch/arbenz/ewp/Lnotes/chapter4.pdf<br>
https://pi.math.cornell.edu/~web6140/TopTenAlgorithms/QRalgorithm.html<br>
Schwarz-Rutishauser Algorithm https://towardsdatascience.com/can-qr-decomposition-be-actually-faster-schwarz-rutishauser-algorithm-a32c0cde8b9b<br>
https://people.inf.ethz.ch/gander/papers/qrneu.pdf

Ejemplos matrices
https://www.intmath.com/matrices-determinants/6-matrices-linear-equations.php

